In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, recall_score, f1_score, classification_report
)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
# plt.rcParams['font.family'] = 'AppleGothic'  # Mac
plt.rcParams['axes.unicode_minus'] = False

# 구조 점수를 계산

## 1) 사고 레이블 추가

_북쪽 : 직접 수집한 이미지      
_부근 : 사고다발지 데이터       

In [27]:
df = pd.read_csv('../CustomVision/accidentlevel.csv')

df1 = df[df.iloc[:, 0].str.endswith('_북쪽.png')].reset_index(drop=True)
df2 = df[~df.iloc[:, 0].str.endswith('_북쪽.png')].reset_index(drop=True)

print(f"df1 (_북쪽.png): {len(df1)}행") # df1 : 수집한 이미지
print(f"df2 (그 외): {len(df2)}행") # df2 : 사고다발지 데이터
display(df1.head())
display(df2.head())

df1 (_북쪽.png): 2247행
df2 (그 외): 1724행


,image,p_wide,p_narrow,p_barrier_yes,p_barrier_no,시설물명,road_width_relative,sidewalk_ratio,parked_density
0,AMI몬테소리손바닥어린이집_북쪽.png,0.660075,0.344503,0.382240,0.614802,AMI몬테소리손바닥어린이집,0.400759,0.000000,1
1,AP어린이집_북쪽.png,0.940672,0.060079,0.822805,0.178012,AP어린이집,0.428197,0.000000,2
2,EPS어린이집_북쪽.png,0.004515,0.995615,0.005522,0.994376,EPS어린이집,0.000000,0.000000,0
3,GC차일드케어어린이집_북쪽.png,0.766128,0.228500,0.616568,0.387656,GC차일드케어어린이집,0.402940,0.042165,0
4,LIG넥스원_어린이집_북쪽.png,0.797194,0.204120,0.853232,0.151019,LIG넥스원_어린이집,0.345928,0.000000,0


,image,p_wide,p_narrow,p_barrier_yes,p_barrier_no,시설물명,road_width_relative,sidewalk_ratio,parked_density
0,강원특별자치도_강릉시_교동(상덕초교_부근)_01.png,0.890403,0.110724,0.512500,0.494360,강원특별자치도_강릉시_교동(상덕초교_부근).png,0.436709,0.264293,0
1,강원특별자치도_강릉시_교동(월드컵사거리_부근)_01.png,0.890403,0.110724,0.512500,0.494360,강원특별자치도_강릉시_교동(월드컵사거리_부근).png,0.436709,0.264293,0
2,강원특별자치도_강릉시_입암동((이안강릉타운아파트앞교차로)_부근)_01.png,0.752523,0.247569,0.476820,0.524749,강원특별자치도_강릉시_입암동((이안강릉타운아파트앞교차로)_부근).png,0.409798,0.000000,0
3,강원특별자치도_강릉시_포남동((송정아파트앞교차로)_부근)_01.png,0.580117,0.422429,0.342686,0.659273,강원특별자치도_강릉시_포남동((송정아파트앞교차로)_부근).png,0.361524,0.000000,3
4,강원특별자치도_강릉시_포남동(강릉시문화센터_부근)_01.png,0.798596,0.203370,0.127873,0.871744,강원특별자치도_강릉시_포남동(강릉시문화센터_부근).png,0.374094,0.202055,0


In [28]:
accident_df1 = pd.read_csv(
    '../preprocessing/raw-data/스쿨존 사고데이터/사고다발지현황.csv')
accident_df2 = pd.read_csv(
    '../preprocessing/raw-data/스쿨존 사고데이터/전국교통사고다발지역표준데이터.csv')

print("=== 사고다발지현황.csv ===")
print(accident_df1.columns.tolist())
print(f"총 {len(accident_df1)}행\n")

print("=== 전국교통사고다발지역표준데이터.csv ===")
print(accident_df2.columns.tolist())
print(f"총 {len(accident_df2)}행")

=== 사고다발지현황.csv ===
['Unnamed: 0', '시군명', '사고년도', '사고유형구분', '다발지식별자', '다발지역그룹식별자', '법정동코드', '위치코드', '시도시군구명', '사고지역위치명', '발생건수', '사상자수', '사망자수', '중상자수', '경상자수', '부상자수', '위도', '경도', '사고다발지역폴리곤정보']
총 2611행

=== 전국교통사고다발지역표준데이터.csv ===
['Unnamed: 0', '사고지역관리번호', '사고연도', '사고유형구분', '위치코드', '사고다발지역시도시군구', '사고지역위치명', '사고건수', '사상자수', '사망자수', '중상자수', '경상자수', '부상신고자수', '위도', '경도', '사고다발지역폴리곤정보', '데이터기준일자', '제공기관코드', '제공기관명']
총 12780행


In [29]:
# 시설물명에서 (지점_부근) 패턴의 지점명만 추출
df2_key = df2['시설물명'].str.replace(r'(_\d+)?\.png$', '', regex=True)
df2_point = df2_key.str.extract(r'\((.+)_부근\)')[0]

names1 = accident_df1['사고지역위치명'].tolist()
names2 = accident_df2['사고지역위치명'].tolist()

def check_match(point, names):
    if pd.isna(point):
        return False
    return any(point in name for name in names)

matched1 = df2_point.apply(check_match, names=names1)
matched2 = df2_point.apply(check_match, names=names2)
matched_either = matched1 | matched2

print(f"df2 전체: {len(df2)}행")
print(f"\n[사고다발지현황]          매칭됨: {matched1.sum()}행 / 안됨: {(~matched1).sum()}행")
print(f"[전국교통사고다발지역]     매칭됨: {matched2.sum()}행 / 안됨: {(~matched2).sum()}행")
print(f"[둘 중 하나라도 매칭]      매칭됨: {matched_either.sum()}행 / 안됨: {(~matched_either).sum()}행")
print("\n=== 둘 다 매칭 안 된 지점명 샘플 (상위 10개) ===")
print(df2_point[~matched_either].unique()[:10].tolist())

df2 전체: 1724행

[사고다발지현황]          매칭됨: 321행 / 안됨: 1403행
[전국교통사고다발지역]     매칭됨: 1634행 / 안됨: 90행
[둘 중 하나라도 매칭]      매칭됨: 1634행 / 안됨: 90행

=== 둘 다 매칭 안 된 지점명 샘플 (상위 10개) ===
['북삼새마을금고_한마음지점', '합동청사_사거리', '광남동_행정복지센터', '우리은행(_남양주지점)', '주공2단지아파트동', '이마트_남양주점', nan, '자연의아이들_탐구몬테소리어린이집', '한전_사거리', '상대원2동_주민센터']


In [30]:
df3 = df2[matched_either].reset_index(drop=True)
print(f"df3 (매칭된 행): {len(df3)}행")
display(df3.head())

df3 (매칭된 행): 1634행


,image,p_wide,p_narrow,p_barrier_yes,p_barrier_no,시설물명,road_width_relative,sidewalk_ratio,parked_density
0,강원특별자치도_강릉시_교동(상덕초교_부근)_01.png,0.890403,0.110724,0.512500,0.494360,강원특별자치도_강릉시_교동(상덕초교_부근).png,0.436709,0.264293,0
1,강원특별자치도_강릉시_교동(월드컵사거리_부근)_01.png,0.890403,0.110724,0.512500,0.494360,강원특별자치도_강릉시_교동(월드컵사거리_부근).png,0.436709,0.264293,0
2,강원특별자치도_강릉시_입암동((이안강릉타운아파트앞교차로)_부근)_01.png,0.752523,0.247569,0.476820,0.524749,강원특별자치도_강릉시_입암동((이안강릉타운아파트앞교차로)_부근).png,0.409798,0.000000,0
3,강원특별자치도_강릉시_포남동((송정아파트앞교차로)_부근)_01.png,0.580117,0.422429,0.342686,0.659273,강원특별자치도_강릉시_포남동((송정아파트앞교차로)_부근).png,0.361524,0.000000,3
4,강원특별자치도_강릉시_포남동(강릉시문화센터_부근)_01.png,0.798596,0.203370,0.127873,0.871744,강원특별자치도_강릉시_포남동(강릉시문화센터_부근).png,0.374094,0.202055,0


In [32]:
df3_point = df3['시설물명'].str.replace(r'(_\d+)?\.png$', '', regex=True).str.extract(r'\((.+)_부근\)')[0]

def get_accident_count(point):
    if pd.isna(point):
        return None
    mask1 = accident_df1['사고지역위치명'].str.contains(point, regex=False)
    if mask1.any():
        return accident_df1.loc[mask1, '발생건수'].iloc[0]
    mask2 = accident_df2['사고지역위치명'].str.contains(point, regex=False)
    if mask2.any():
        return accident_df2.loc[mask2, '사고건수'].iloc[0]
    return None

df3['발생건수'] = df3_point.apply(get_accident_count)
print(f"발생건수 추가 완료: {df3['발생건수'].notna().sum()} / {len(df3)}행")
display(df3.head())

발생건수 추가 완료: 1634 / 1634행


,image,p_wide,p_narrow,p_barrier_yes,p_barrier_no,시설물명,road_width_relative,sidewalk_ratio,parked_density,발생건수
0,강원특별자치도_강릉시_교동(상덕초교_부근)_01.png,0.890403,0.110724,0.512500,0.494360,강원특별자치도_강릉시_교동(상덕초교_부근).png,0.436709,0.264293,0,2
1,강원특별자치도_강릉시_교동(월드컵사거리_부근)_01.png,0.890403,0.110724,0.512500,0.494360,강원특별자치도_강릉시_교동(월드컵사거리_부근).png,0.436709,0.264293,0,3
2,강원특별자치도_강릉시_입암동((이안강릉타운아파트앞교차로)_부근)_01.png,0.752523,0.247569,0.476820,0.524749,강원특별자치도_강릉시_입암동((이안강릉타운아파트앞교차로)_부근).png,0.409798,0.000000,0,3
3,강원특별자치도_강릉시_포남동((송정아파트앞교차로)_부근)_01.png,0.580117,0.422429,0.342686,0.659273,강원특별자치도_강릉시_포남동((송정아파트앞교차로)_부근).png,0.361524,0.000000,3,3
4,강원특별자치도_강릉시_포남동(강릉시문화센터_부근)_01.png,0.798596,0.203370,0.127873,0.871744,강원특별자치도_강릉시_포남동(강릉시문화센터_부근).png,0.374094,0.202055,0,3
